# RouteTune Phase 3: verified QLoRA execution
Start from a freshly created Colab NVIDIA GPU runtime (T4 or better); do not reuse a runtime where Transformers was imported or monkey-patched. Run through the install cell, which deliberately restarts the Python process, then reconnect and continue at **Post-restart verification**. Do not reload modules or patch `PreTrainedConfig`: only a process restart guarantees that the installed dependency lock is active. Upload a ZIP of the current project. The notebook does not assume the GitHub repository is public. Training and held-out evaluation fail closed when evidence is missing.

In [ ]:
import platform, subprocess, sys
print(sys.version)
print(platform.platform())
subprocess.run(["nvidia-smi"], check=True)

Upload one project ZIP made from the current `D:\\Fun\\RouteTune` worktree. This extracts only paths contained by the destination and selects the single directory containing `pyproject.toml`.

In [ ]:
from google.colab import files
from pathlib import Path
import os, zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one RouteTune project ZIP")
archive = Path("/content") / next(iter(uploaded))
source_root = Path("/content/routetune-source")
source_root.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as bundle:
    for member in bundle.infolist():
        destination = (source_root / member.filename).resolve()
        if not destination.is_relative_to(source_root.resolve()):
            raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
    bundle.extractall(source_root)
projects = [path.parent for path in source_root.rglob("pyproject.toml")]
if len(projects) != 1:
    raise RuntimeError(f"Expected one project root, found {projects}")
PROJECT = projects[0]
os.chdir(PROJECT)
print(PROJECT)

In [ ]:
import signal
install_marker = PROJECT / ".colab-dependencies-installed"
if install_marker.exists():
    raise RuntimeError("Dependency installation already completed; continue at Post-restart verification")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--index-url", "https://download.pytorch.org/whl/cu130", "torch==2.14.1", "torchvision==0.29.1"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-e", ".[test,gpu]"],
    check=True,
)
install_marker.write_text("installed; restart required\n", encoding="utf-8")
print("Dependency lock installed. The runtime will restart now; reconnect and continue at Post-restart verification.", flush=True)
os.kill(os.getpid(), signal.SIGKILL)

## Post-restart verification
After Colab reconnects, start here. This rediscovers the extracted project in the new process and verifies exact package versions, the complete real TRL `SFTConfig` (including 10% fractional warm-up) and `SFTTrainer` APIs, the absence of `_commit_hash` injection, CUDA, and the complete offline suite before any model metadata or weights are requested. `verify_gpu_environment()` is the real-library configuration preflight; do not continue if it fails.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
projects = [path.parent for path in Path("/content/routetune-source").rglob("pyproject.toml")]
if len(projects) != 1:
    raise RuntimeError(f"Expected one installed project root after restart, found {projects}")
PROJECT = projects[0]
os.chdir(PROJECT)
from routetune.training import assess_pip_check, environment_report, verify_gpu_environment
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True, capture_output=True)
pip_check_output = "\n".join(part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip())
print(pip_check_output or "(pip check produced no output)")
DEPENDENCY_READINESS = assess_pip_check(pip_check_output, pip_check.returncode)
print(json.dumps(DEPENDENCY_READINESS, indent=2, sort_keys=True))
if not DEPENDENCY_READINESS["ready"]:
    raise RuntimeError("Required dependency conflicts block RouteTune execution")
print(json.dumps(environment_report(), indent=2, sort_keys=True))
print(verify_gpu_environment())
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

Use Google Drive for durable run evidence. If the runtime is interrupted before the adapter is saved, the manifest remains `RUNNING` and must not be presented as completed or silently resumed.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CACHE = Path("/content/routetune-data")
RUNS = Path("/content/drive/MyDrive/RouteTune/phase3-runs")
RUNS.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "routetune.data", "--cache-dir", str(CACHE)], check=True)
from routetune.experiment import atomic_json_write, update_run
from routetune.training import prepare_phase3_run
RUN = prepare_phase3_run(CACHE, RUNS, offline=True)
atomic_json_write(RUN / "dependency-readiness.json", DEPENDENCY_READINESS)
update_run(RUN, artifacts={"dependency_readiness": "dependency-readiness.json"})
print(RUN)

In [ ]:
from routetune.training import execute_phase3_training
execute_phase3_training(RUN, CACHE)

In [ ]:
from routetune.experiment import load_manifest
import json
manifest = load_manifest(RUN)
print(json.dumps(manifest, indent=2, sort_keys=True))
for name in ("untuned-validation.json", "adapted-validation.json"):
    path = RUN / name
    if path.exists():
        result = json.loads(path.read_text())
        print(name, json.dumps(result["metrics"], indent=2))

## One-time official held-out gate
Leave this disabled until the manifest above is `VERIFIED` and you accept consuming the single official held-out evaluation. The function atomically claims the gate before reading predictions, and refuses a rerun.

In [ ]:
RUN_FINAL_HELDOUT = False
if RUN_FINAL_HELDOUT:
    from routetune.training import execute_final_heldout
    result_path = execute_final_heldout(RUN, CACHE)
    print(result_path.read_text())
else:
    print("Held-out gate not consumed.")

In [ ]:
import shutil
export = shutil.make_archive(f"/content/{RUN.name}", "zip", RUN.parent, RUN.name)
files.download(export)